<a href="https://colab.research.google.com/github/yamms2340/researchWorkCodes/blob/main/calcLyponoClassical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

def calculate_lyapunov_from_csv(file_path, r=28.0, sigma=10.0, b=2.667, dt=0.01):
    """
    Reads a CSV file containing trajectory data and calculates the Lyapunov spectrum.
    Assumes the first three columns are x, y, and z respectively.
    """
    print(f"Loading data from '{file_path}'...")

    # Load the CSV. pandas handles headers and weird formatting automatically.
    df = pd.read_csv(file_path)

    # Extract the first 3 columns as our (x, y, z) coordinates as a NumPy array
    u = df.iloc[:, :3].values

    n = len(u)
    q = np.eye(3)
    le = np.zeros(3)

    print(f"Calculating Lyapunov spectrum across {n} time steps...")

    for i in range(n):
        x, y, z = u[i]

        # Evaluate the Lorenz Jacobian matrix at the current state
        jac = np.array([
            [-sigma, sigma, 0],
            [r-z, -1, -x],
            [y, x, -b]
        ])

        # Evolve the perturbation matrix
        dq = np.dot(jac, q) * dt
        q = q + dq

        # Apply Gram-Schmidt orthonormalization
        q, r_mat = np.linalg.qr(q)

        # Accumulate the log of the diagonal elements
        le += np.log(np.abs(np.diag(r_mat)) + 1e-12)

    # Average over total integration time
    final_le = le / (n * dt)
    return final_le

# --- Execution Example ---
if __name__ == "__main__":
    # Replace this with the exact name of the file you upload!
    uploaded_filename = "lorenz_data.csv"

    try:
        # Note: Make sure 'r', 'sigma', 'b', and 'dt' match the parameters
        # you used in your own data generation notebook!
        true_le = calculate_lyapunov_from_csv(
            file_path=uploaded_filename,
            r=28.0,
            sigma=10.0,
            b=2.667,
            dt=0.01
        )

        print("\n--- Final Lyapunov Spectrum ---")
        print(f"LE1 (Maximal):     {true_le[0]:.4f}")
        print(f"LE2 (Marginal):    {true_le[1]:.4f}")
        print(f"LE3 (Contractive): {true_le[2]:.4f}")

        # --- ADDED SAVING CODE HERE ---
        # Note: Changed 't_le' to 'true_le' to match the variable in this script
        df_classical = pd.DataFrame({
            "LE_Type": ["LE1 (Max)", "LE2 (Marg)", "LE3 (Cont)"],
            "Classical_Value": true_le
        })
        df_classical.to_csv("classical_results.csv", index=False)
        print("Successfully saved to 'classical_results.csv'")
        # ------------------------------

    except FileNotFoundError:
        print(f"Error: Could not find '{uploaded_filename}'. Check the file path.")
    except Exception as e:
        print(f"An error occurred: {e}")

Loading data from 'lorenz_data.csv'...
Calculating Lyapunov spectrum across 1000 time steps...

--- Final Lyapunov Spectrum ---
LE1 (Maximal):     0.9066
LE2 (Marginal):    -0.0821
LE3 (Contractive): -14.8496
Successfully saved to 'classical_results.csv'
